# Explore a BERTrend model
The purpose of this notebook is to show how to load an existing BERTrend model and explore its content. 

BERTrend is a tool for analyzing trends and detecting weak signals in text data over time. It builds upon BERTopic by training multiple models over different time periods and merging them to track topic evolution.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

from bertrend.BERTrend import BERTrend
from bertrend import MODELS_DIR, CACHE_PATH
from bertrend.trend_analysis.visualizations import (
    create_sankey_diagram_plotly, 
    create_topic_size_evolution_figure,
    plot_newly_emerged_topics,
    plot_num_topics
)
from bertrend.trend_analysis.weak_signals import analyze_signal
from bertrend.BERTopicModel import BERTopicModel

## 1. Loading the BERTrend model

By default, BERTrend models are saved in the `MODELS_DIR` (which is typically `.bertrend/cache/models`). 
You can load a model using the `BERTrend.restore_model(models_path)` class method.

In [ ]:
# Define the path to your saved model
models_path = MODELS_DIR

try:
    bertrend = BERTrend.restore_model(models_path)
    print(f"Successfully loaded BERTrend model from {models_path}")
except FileNotFoundError:
    print(f"No model found at {models_path}. \nCreating a small sample model for demonstration purposes...")
    
    # --- DUMMY TRAINING START ---
    # This part is just to make the notebook runnable if you don't have a model yet.
    from bertrend.utils.data_loading import load_data
    
    # Load some sample data (covid news)
    data_path = Path("data/bertopic/covid_news.csv")
    if data_path.exists():
        df = load_data(data_path).head(500) # Small sample for speed
        df["date"] = pd.to_datetime(df["date"])
        
        # Initialize and fit a small model
        tm = BERTopicModel()
        bertrend = BERTrend(topic_model=tm)
        bertrend.fit(df, column="content", date_column="date")
        
        # Save it so we can practice loading next time
        bertrend.save_model(models_path)
    else:
        print("Sample data not found. Please provide a path to a saved model.")
        bertrend = None
    # --- DUMMY TRAINING END ---

## 2. Basic Content Exploration

Once the model is loaded, we can explore several important dataframes that summarize the topic analysis.

In [ ]:
if bertrend:
    print("### Last Topic Model Info")
    # The last trained topic model
    display(bertrend.last_topic_model.get_topic_info().head(10))

    print("\n### Merged Topics DataFrame")
    # Contains the combined information of all merged topics over time
    if bertrend.merged_df is not None:
        display(bertrend.merged_df.head(10))
    
    print("\n### All New Topics")
    # Newly emerged topics at each time period
    if bertrend.all_new_topics_df is not None:
        display(bertrend.all_new_topics_df.head(10))

## 3. Visualizing Topic Evolution

BERTrend provides several visualizations to understand how topics change over time.

In [ ]:
if bertrend and bertrend.all_merge_histories_df is not None:
    print("### Topic Merging (Sankey Diagram)")
    # Visualizes how topics from different time periods were merged together
    fig_sankey = create_sankey_diagram_plotly(bertrend.all_merge_histories_df)
    fig_sankey.show()

if bertrend and bertrend.all_new_topics_df is not None:
    print("\n### Newly Emerged Topics")
    fig_new = plot_newly_emerged_topics(bertrend.all_new_topics_df)
    fig_new.show()

## 4. Weak Signal Analysis

One of the key features of BERTrend is detecting weak signals. You can analyze the signal for a specific topic.

In [ ]:
if bertrend:
    # Pick a topic ID to analyze (e.g., topic 0)
    target_topic = 0
    
    # We need a reference date for the analysis (usually the latest period in the model)
    periods = bertrend.get_periods()
    if periods:
        selected_timestamp = max(periods)
        
        # analyze_signal returns a summary and a detailed signal analysis
        summary, analysis = analyze_signal(bertrend, topic_number=target_topic, current_date=selected_timestamp)
        
        if analysis:
            print(f"### Signal Analysis for Topic {target_topic} at {selected_timestamp.date()}")
            print(f"\nShort term implications:")
            for impl in analysis.potential_implications.short_term_implications:
                print(f"- {impl}")
            
            print(f"\nDrivers:")
            for driver in analysis.drivers_inhibitors.drivers:
                print(f"- {driver}")
        else:
            print(f"Could not analyze signal for topic {target_topic} at {selected_timestamp.date()}")
    
    # Visualize the popularity evolution of topics
    if bertrend.topic_sizes:
        fig_evolution = create_topic_size_evolution_figure(bertrend.topic_sizes)
        fig_evolution.show()

## 5. Topic Deep Dive

You can inspect the representative documents and generated descriptions for a topic.

In [ ]:
if bertrend:
    topic_id = 0
    topic_info = bertrend.last_topic_model.get_topic_info()
    topic_row = topic_info[topic_info['Topic'] == topic_id]
    
    if not topic_row.empty:
        print(f"### Representative Documents for Topic {topic_id}")
        repr_docs = topic_row['Representative_Docs'].values[0]
        for i, doc in enumerate(repr_docs[:3]):
            print(f"\nDoc {i+1}: {doc[:300]}...")